In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchaudio

from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import librosa
import ast
import os
from typing import Dict, List
import subprocess




# from phoneme_encoder import *
# from tajweed_eval import *

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
# !mkdir -p /content/kaggle/working
# !unzip "/content/drive/MyDrive/archive.zip" -d /content/kaggle/working/
# !ln -s /content/kaggle kaggle
# print(os.path.exists("kaggle/working/Hudhaify/audio/Hudhaify_64kbps/086006.mp3"))

In [7]:
MODEL_PATH = "tajweed_ctc_model.pth"
DATASET_PATH='../../datasets/Quran_ds/audio/audio'
TRAIN_DS_PATH='quran_train.csv'
TEST_DS_PATH='quran_test.csv'


# MODEL_PATH='drive/MyDrive/tajweed_ctc_model.pth'
# DATASET_PATH='../../datasets/Quran_ds/audio/audio'
# TRAIN_DS_PATH='quran_train.csv'
# TEST_DS_PATH='quran_test.csv'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 4
sr = 16000


device


device(type='cuda')

In [5]:
SPACE_TOKEN = "<space>"
WASL = "<wasl>"

SHORT_VOWELS: List[str] = ["a", "i", "u"]
LONG_VOWELS: List[str] = ["aa", "ii", "uu"]
TANWEEN: List[str] = ["an", "in", "un"]

CONSONANTS: List[str] = [
    "ʔ",
    "ʔw",
    "b",
    "t",
    "θ",
    "j",
    "ħ",
    "x",
    "d",
    "ð",
    "r",
    "z",
    "s",
    "ʃ",
    "sˤ",
    "dˤ",
    "tˤ",
    "ðˤ",
    "ʕ",
    "ɣ",
    "f",
    "q",
    "k",
    "l",
    "m",
    "n",
    "h",
    "w",
    "y",
]

CV_SHORT: List[str] = [c + v for c in CONSONANTS for v in SHORT_VOWELS]
CV_LONG: List[str] = [c + v for c in CONSONANTS for v in LONG_VOWELS]
CV_TANWEEN: List[str] = [c + v for c in CONSONANTS for v in TANWEEN]
CONSONANTS: List[str] = [
    "ʔ",
    "ʔw",
    "b",
    "t",
    "θ",
    "j",
    "ħ",
    "x",
    "d",
    "ð",
    "r",
    "z",
    "s",
    "ʃ",
    "sˤ",
    "dˤ",
    "tˤ",
    "ðˤ",
    "ʕ",
    "ɣ",
    "f",
    "q",
    "k",
    "l",
    "m",
    "n",
    "h",
    "w",
    "y",
]
SHORT_VOWELS: List[str] = ["a", "i", "u"]
LONG_VOWELS: List[str] = ["aa", "ii", "uu"]
TANWEEN: List[str] = ["an", "in", "un"]
BLANK_TOKEN = "<blank>"




PHONEMES: List[str] = (
    ["sil", SPACE_TOKEN, WASL] + CONSONANTS + CV_SHORT + CV_LONG + CV_TANWEEN
)
IDX2PHONEME: Dict[int, str] = {i: p for i, p in enumerate(PHONEMES)}

PHONEMES_CTC: List[str] = [BLANK_TOKEN] + PHONEMES
phoneme_to_id: Dict[str, int] = {p: i for i, p in enumerate(PHONEMES_CTC)}
blank_id: int = phoneme_to_id[BLANK_TOKEN]


### تسكين الحرف عند الوقف و همزة الوصل عند بدء الكلام

### فأذلهما الشيطان   (المد عند الوصل)

### add silent char

### add selah to madd eval

### لِأُولِي  أَنَا

# بِأَييِّكُمُ

# بِاللَّهِ   إِنَّمَا يَعْمُرُ مَسَاجِدَ اللَّٰهِ مَنْ آمَنَ بِاللَّهِ

In [6]:
# audio_dir = '../../datasets/Quran_ds/audio/audio'
# text_file = "quran-simple_clean.txt"
# output_csv = "final_dataset.csv"

# data= {}
# quran_data={}

# for folders  , s , l in os.walk(audio_dir):
#     if(l.__len__()==0):
#         continue

#     reciter_name=folders.split('/')[-1]
#     data[reciter_name]=l

# with open(text_file, "r", encoding="utf-8") as f:
#     for line in f:
#         l=line.split('|')
#         if(len(l)!=3):
#             continue

#         surah=line.split('|')[0]
#         ayah= line.split('|')[1]

#         if(len(surah)==1):
#             surah= f'00{surah}'
#         elif(len(surah)==2):
#             surah='0'+surah

#         if(len(ayah)==1):
#             ayah= f'00{ayah}'
#         elif(len(ayah)==2):
#             ayah='0'+ayah

#         key=f'{surah+ayah}'
#         quran_data[key]=line.split('|')[2].replace('\n','')

# final_data=[]

# for reciter in data:
#     files= data[reciter]
#     for file in files:

        
        
#         if(quran_data.get(file.split('.')[0])):
#             # print(reciter)
#             # print(file.split('.')[0])
#             # print(quran_data.get(file.split('.')[0]))

#             if(file[3]=='000' or file[3:]=='001'):
#                 continue
            
#             item={
#                 "surah":file[:3],
#                 "ayah":file[3:6],
#                 'path_of_audio':reciter+'/'+file,
#                 'reciter_name':reciter,
#                 "ayah_text":quran_data.get(file.split('.')[0]),
#             }
#             final_data.append(item)
    

# # Create DataFrame
# df = pd.DataFrame(final_data)

# # Save CSV
# df.to_csv('quran_ds.csv', index=False, encoding="utf-8")

# print("CSV created:", output_csv)

In [ ]:
# df = pd.read_csv("quran_train_mini.csv")

# total_seconds = 0
# failed_files = []

# for path in df["path_of_audio"]:
#     try:
#         info = torchaudio.info(DATASET_PATH+'/'+ path)
#         total_seconds += info.num_frames / info.sample_rate
#     except Exception:
        
#         failed_files.append(path)

# total_hours = total_seconds / 3600

# print(f"Total duration: {total_hours:.2f} hours")
# print(f"Failed files: {len(failed_files)}")


/home/mahmoud-bannan/miniconda3/envs/torch_mnb/lib/python3.9/site-packages/torchaudio/_backend/soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/closed ones). You may otherwise ignore this warning.
  warnings.warn(


Total duration: 48.29 hours
Failed files: 0


### Total duration (TRAIN): 146.90 hours

### Total duration  (TEST): 10.91 hours

In [8]:
# import importlib
# import phoneme_encoder

# importlib.reload(phoneme_encoder)
# phoneme_encoder.test_encoder()

In [9]:
# import importlib
# import phoneme_encoder

# importlib.reload(phoneme_encoder)

# ph, meta = phoneme_encoder.text_to_phonemes_with_mapping("بِسْمِ اللَّٰهِ الرَّحْمَـٰنِ الرَّحِيمِ الم",for_debug=False)

# ph

In [10]:
# import generate_quran_phoneme
# import importlib

# importlib.reload(generate_quran_phoneme)

# generate_quran_phoneme.generate_phoneme_dataset()

# generate_quran_phoneme.split_ds()

In [11]:
# from phoneme_encoder import  text_to_phonemes_with_mapping
# df= pd.read_csv('quran_test.csv')
# phonemes=[]
# for text in df['ayah_text']:
#     phonemes.append(text_to_phonemes_with_mapping(text=text,for_debug=False))

# df['phonemes']=phonemes
# df.head()
# df.to_csv('quran_test.csv')

In [12]:
def add_noise(signal, noise_level=0.005):
    noise = np.random.randn(len(signal))
    return signal + noise_level * noise

def random_gain(signal):
    gain = np.random.uniform(0.8, 1.2)
    return signal * gain

def spec_augment(features, time_mask=10, freq_mask=5):
    f = features.copy()

    # Time masking
    t = np.random.randint(0, f.shape[0] - time_mask)
    f[t:t+time_mask, :] = 0

    # Frequency masking
    f0 = np.random.randint(0, f.shape[1] - freq_mask)
    f[:, f0:f0+freq_mask] = 0

    return f

def convert_to_wav(audio_path):
    wav_path = audio_path.replace(".mp3", ".wav")
    subprocess.run(["ffmpeg", "-i", audio_path, "-ar", "16000", "-ac", "1", wav_path, "-y"], 
                   capture_output=True)
    return wav_path

In [ ]:
def extract_features(audio_path, sr=16000,training=True):
    
    if audio_path.endswith(".mp3"):
        audio_path = convert_to_wav(audio_path)

    signal, sr = librosa.load(audio_path, sr=sr)
    
    if training and np.random.rand() < 0.5:
        signal = add_noise(signal, noise_level=0.003)

    if training and np.random.rand() < 0.5:
        signal = random_gain(signal)

    mfcc = librosa.feature.mfcc(
        y=signal, sr=sr, n_mfcc=13, n_fft=int(0.025 * sr), hop_length=int(0.010 * sr)
    )

    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    features = np.vstack([mfcc, delta, delta2]).T

    if training and np.random.rand() < 0.5:
        features = spec_augment(features)

    # CMVN
    mean = np.mean(features, axis=0)
    std = np.std(features, axis=0) + 1e-8
    features = (features - mean) / std

    return features

In [ ]:
class TajweedCTCDataset(Dataset):

    def __init__(self, dataframe , training=True):
        self.df = dataframe
        self.training=training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # Extract features
        features = extract_features(DATASET_PATH+"/"+row["path_of_audio"],self.training)

        # Convert text → phoneme sequence
        phoneme_seq, _ = ast.literal_eval(row['phonemes'])

        target_ids = [phoneme_to_id[p] for p in phoneme_seq if p in phoneme_to_id]

        return (
            torch.tensor(features, dtype=torch.float32),
            torch.tensor(target_ids, dtype=torch.long),
            features.shape[0],
            len(target_ids),
        )

In [15]:
def ctc_collate(batch):

    features, targets, input_lengths, target_lengths = zip(*batch)

    padded_features = pad_sequence(features, batch_first=True)

    concatenated_targets = torch.cat(targets)

    return (
        padded_features,
        concatenated_targets,
        torch.tensor(input_lengths),
        torch.tensor(target_lengths),
    )

In [ ]:
class BiLSTM_CTC(nn.Module):

    def __init__(self, input_dim, hidden_dim, num_layers, num_classes):
        super().__init__()

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3,
        )

        self.dropout = nn.Dropout(p=0.3)  

        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        packed_out, _ = self.lstm(packed)

        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        
        outputs = self.dropout(outputs)            

        logits = self.classifier(outputs)

        logits = logits.permute(1, 0, 2)

        return logits

In [17]:
train_df = pd.read_csv(TRAIN_DS_PATH)
val_df = pd.read_csv(TEST_DS_PATH)

In [18]:
train_dataset = TajweedCTCDataset(train_df)
val_dataset = TajweedCTCDataset(val_df)

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, collate_fn=ctc_collate
)

val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, collate_fn=ctc_collate
)

In [19]:
next(enumerate(train_loader))

(0,
 (tensor([[[-2.3751, -2.6396,  0.3431,  ...,  0.2898, -0.3497, -0.5337],
           [-2.3751, -2.6396,  0.3431,  ...,  0.2898, -0.3497, -0.5337],
           [-2.3751, -2.6396,  0.3431,  ...,  0.2898, -0.3497, -0.5337],
           ...,
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
  
          [[-2.4436, -2.1740, -0.4602,  ...,  1.5786, -0.4394, -0.8161],
           [-2.1108, -2.6624, -0.4204,  ...,  1.5786, -0.4394, -0.8161],
           [-2.1218, -2.1171,  0.0047,  ...,  1.5786, -0.4394, -0.8161],
           ...,
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
  
          [[-2.8554, -2.7673,  0.7685,  ..., -1.0224, -1.8299, -1.6103],
       

In [20]:
model = BiLSTM_CTC(
    input_dim=39, hidden_dim=256, num_layers=3, num_classes=len(phoneme_to_id)
).to(device)

In [21]:
model.state_dict

<bound method Module.state_dict of BiLSTM_CTC(
  (lstm): LSTM(39, 256, num_layers=3, batch_first=True, dropout=0.3, bidirectional=True)
  (classifier): Linear(in_features=512, out_features=294, bias=True)
)>

In [22]:
ctc_loss = nn.CTCLoss(blank=blank_id, zero_infinity=True, reduction="mean")
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [23]:
# checkpoint = torch.load(model_path, map_location=torch.device(device))
# model.load_state_dict(checkpoint)

In [24]:
def ctc_decode(frame_preds, blank_id):

    decoded = []
    prev = None

    for p in frame_preds:

        p = int(p)

        if p != blank_id and p != prev:
            decoded.append(p)

        prev = p

    return decoded

In [25]:
best_acc_loss = float("inf")


def train_model(model, train_loader, val_loader, epochs=20):

    global best_acc_loss

    for epoch in range(epochs):

        # =========================
        # TRAIN
        # =========================
        model.train()
        train_loss = 0

        for features, targets, input_lengths, target_lengths in tqdm(train_loader):

            features = features.to(device)
            targets = targets.to(device)
            input_lengths = input_lengths.to(device)
            target_lengths = target_lengths.to(device)

            optimizer.zero_grad()

            logits = model(features, input_lengths)

            log_probs = torch.log_softmax(logits, dim=2)

            loss = ctc_loss(
                log_probs,
                targets,
                input_lengths,
                target_lengths,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        # =========================
        # VALIDATION
        # =========================
        model.eval()
        val_loss = 0

        with torch.no_grad():

            for features, targets, input_lengths, target_lengths in tqdm(val_loader):

                features = features.to(device)
                targets = targets.to(device)
                input_lengths = input_lengths.to(device)
                target_lengths = target_lengths.to(device)

                logits = model(features, input_lengths)

                log_probs = torch.log_softmax(logits, dim=2)

                loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)

                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)

        # =========================
        # SAVE BEST MODEL
        # =========================
        if avg_val_loss < best_acc_loss:
            best_acc_loss = avg_val_loss
            torch.save(model.state_dict(), MODEL_PATH)
            print("✅ Model saved")

        # =========================
        # LIVE DECODING DIAGNOSTIC
        # =========================
        with torch.no_grad():

            features, targets, input_lengths, target_lengths = next(iter(val_loader))

            features = features.to(device)
            input_lengths = input_lengths.to(device)

            logits = model(features, input_lengths)

            pred = torch.argmax(logits, dim=2)  # (T,B)

            decoded = ctc_decode(pred[:, 0], blank_id)

            tokens = [IDX2PHONEME[i] for i in decoded]

            print("Sample prediction:")
            print(tokens)

        # =========================
        # LOGGING
        # =========================
        
        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}")
        print(f"Val Loss:   {avg_val_loss:.4f}")
        print("-" * 40)

In [26]:
train_model(model=model,train_loader=train_loader,val_loader=val_loader,epochs=1)

  0%|          | 0/3741 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 186.00 MiB. GPU 

In [ ]:
# def phoneme_alignment(ref, hyp):

#     n = len(ref)
#     m = len(hyp)

#     dp = [[0] * (m + 1) for _ in range(n + 1)]

#     for i in range(n + 1):
#         dp[i][0] = i
#     for j in range(m + 1):
#         dp[0][j] = j

#     for i in range(1, n + 1):
#         for j in range(1, m + 1):

#             if ref[i - 1] == hyp[j - 1]:
#                 cost = 0
#             else:
#                 cost = 1

#             dp[i][j] = min(
#                 dp[i - 1][j] + 1,  # deletion
#                 dp[i][j - 1] + 1,  # insertion
#                 dp[i - 1][j - 1] + cost,  # substitution
#             )

#     return dp

In [ ]:
# def extract_errors(ref, hyp):

#     dp = phoneme_alignment(ref, hyp)

#     i = len(ref)
#     j = len(hyp)

#     errors = []

#     while i > 0 or j > 0:

#         if i > 0 and j > 0 and ref[i - 1] == hyp[j - 1]:
#             i -= 1
#             j -= 1

#         elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
#             errors.append(
#                 {"type": "Deletion", "phoneme": ref[i - 1], "position": i - 1}
#             )
#             i -= 1

#         elif j > 0 and dp[i][j] == dp[i][j - 1] + 1:
#             errors.append(
#                 {"type": "Insertion", "phoneme": hyp[j - 1], "position": j - 1}
#             )
#             j -= 1

#         else:
#             errors.append(
#                 {
#                     "type": "Substitution",
#                     "expected": ref[i - 1],
#                     "predicted": hyp[j - 1],
#                     "position": i - 1,
#                 }
#             )
#             i -= 1
#             j -= 1

#     return errors[::-1]

In [ ]:
# def greedy_decode_with_frames(logits, blank_id):
#     """
#     logits: (T, batch, num_classes)
#     Returns:
#         frame_preds: list of predicted phoneme ids per frame
#         collapsed: phoneme sequence after CTC collapse
#     """

#     pred = torch.argmax(logits, dim=2)  # (T, batch)
#     pred = pred[:, 0].cpu().numpy()  # assume batch size 1 for now

#     frame_preds = pred.tolist()

#     collapsed = []
#     prev = None

#     for p in frame_preds:
#         if p != blank_id and p != prev:
#             collapsed.append(p)
#         prev = p

#     return frame_preds, collapsed

In [ ]:
# def frames_to_segments(frame_preds, blank_id):

#     segments = []
#     current_ph = None
#     count = 0

#     for p in frame_preds:

#         if p == blank_id:
#             if current_ph is not None:
#                 segments.append((current_ph, count))
#                 current_ph = None
#                 count = 0
#             continue

#         if p != current_ph:
#             if current_ph is not None:
#                 segments.append((current_ph, count))
#             current_ph = p
#             count = 1
#         else:
#             count += 1

#     if current_ph is not None:
#         segments.append((current_ph, count))

#     return segments

In [ ]:
# def normalize_segment_durations(segments):
#     """
#     segments: list of (phoneme_id, frame_count)
#     Returns list of (phoneme_id, frame_count, normalized_duration)
#     """

#     total_frames = sum(frames for _, frames in segments)
#     num_segments = len(segments)

#     if num_segments == 0:
#         return []

#     avg_duration = total_frames / num_segments

#     normalized = []

#     for phoneme_id, frames in segments:
#         norm = frames / avg_duration
#         normalized.append((phoneme_id, frames, norm))

#     return normalized

In [ ]:
# def align_phonemes(expected, predicted):
#     """
#     Align expected phoneme sequence with predicted sequence.

#     Returns list of tuples:
#     (expected_phoneme, predicted_phoneme)
#     """

#     n = len(expected)
#     m = len(predicted)

#     # DP table
#     dp = [[0] * (m + 1) for _ in range(n + 1)]

#     for i in range(n + 1):
#         dp[i][0] = i

#     for j in range(m + 1):
#         dp[0][j] = j

#     for i in range(1, n + 1):
#         for j in range(1, m + 1):

#             if expected[i - 1] == predicted[j - 1]:
#                 cost = 0
#             else:
#                 cost = 1

#             dp[i][j] = min(
#                 dp[i - 1][j] + 1,  # deletion
#                 dp[i][j - 1] + 1,  # insertion
#                 dp[i - 1][j - 1] + cost,  # substitution
#             )

#     # Backtrack
#     alignment = []
#     i = n
#     j = m

#     while i > 0 and j > 0:

#         if expected[i - 1] == predicted[j - 1]:
#             alignment.append((expected[i - 1], predicted[j - 1]))
#             i -= 1
#             j -= 1

#         elif dp[i][j] == dp[i - 1][j - 1] + 1:
#             alignment.append((expected[i - 1], predicted[j - 1]))
#             i -= 1
#             j -= 1

#         elif dp[i][j] == dp[i - 1][j] + 1:
#             alignment.append((expected[i - 1], None))
#             i -= 1

#         else:
#             alignment.append((None, predicted[j - 1]))
#             j -= 1

#     alignment.reverse()

#     return alignment

In [ ]:
# def evaluate_madd_rules(rules, segments, tolerance=0.8):

#     avg_short = compute_avg_short_duration(segments)

#     results = []
#     errors = []

#     if avg_short is None:
#         return results, errors

#     for rule in rules:

#         phoneme_index = rule["phoneme_index"]
#         expected = rule["expected_harakat"]

#         if phoneme_index >= len(segments):
#             continue

#         pid, frames = segments[phoneme_index]

#         observed = frames / avg_short

#         if abs(observed - expected) <= tolerance:
#             status = "Correct"
#         elif observed < expected:
#             status = "Too Short"
#         else:
#             status = "Too Long"

#         result = {
#             "phoneme_index": phoneme_index,
#             "type": rule["type"],
#             "expected": expected,
#             "observed": round(observed, 2),
#             "status": status,
#         }

#         results.append(result)

#         if status != "Correct":
#             errors.append(result)

#     return results, errors

In [ ]:
# def word_errors(expected_words, predicted_words):

#     n = len(expected_words)
#     m = len(predicted_words)

#     dp = [[0] * (m + 1) for _ in range(n + 1)]

#     for i in range(n + 1):
#         dp[i][0] = i
#     for j in range(m + 1):
#         dp[0][j] = j

#     for i in range(1, n + 1):
#         for j in range(1, m + 1):

#             if expected_words[i - 1] == predicted_words[j - 1]:
#                 cost = 0
#             else:
#                 cost = 1

#             dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)

#     # Backtrace
#     i, j = n, m
#     errors = []

#     while i > 0 or j > 0:

#         if i > 0 and j > 0 and expected_words[i - 1] == predicted_words[j - 1]:
#             i -= 1
#             j -= 1

#         elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
#             errors.append(
#                 {
#                     "type": "Missing Word",
#                     "word": expected_words[i - 1],
#                     "position": i - 1,
#                 }
#             )
#             i -= 1

#         elif j > 0 and dp[i][j] == dp[i][j - 1] + 1:
#             errors.append(
#                 {
#                     "type": "Extra Word",
#                     "word": predicted_words[j - 1],
#                     "position": j - 1,
#                 }
#             )
#             j -= 1

#         else:
#             errors.append(
#                 {
#                     "type": "Wrong Word",
#                     "expected": expected_words[i - 1],
#                     "predicted": predicted_words[j - 1],
#                     "position": i - 1,
#                 }
#             )
#             i -= 1
#             j -= 1

#     return errors[::-1]

In [ ]:
# def evaluate_audio(model, audio_path, ayah_text):

#     model.eval()

#     # ---------------------------------------
#     # 1) Extract audio features
#     # ---------------------------------------

#     features = extract_features(audio_path)

#     features = torch.from_numpy(features).float()

#     input_len = torch.tensor([features.shape[0]], dtype=torch.long)

#     features = features.unsqueeze(0).to(device)

#     # ---------------------------------------
#     # 2) Run ASR model
#     # ---------------------------------------

#     with torch.no_grad():

#         logits = model(features, input_len)  # (T,B,V)
#         frame_preds = torch.argmax(logits, dim=-1)  # (T,B)

#     frame_preds = frame_preds.squeeze(1).cpu().numpy()  # (T,)

#     # ---------------------------------------
#     # 3) Convert frames → phoneme segments
#     # ---------------------------------------

#     segments = frames_to_segments(frame_preds, blank_id)

#     # segments example:
#     # [(pid, frames), (pid, frames), ...]

#     # ---------------------------------------
#     # 4) Predicted phoneme sequence
#     # ---------------------------------------

#     predicted_phonemes = [IDX2PHONEME[pid] for pid, _ in segments]

#     # ---------------------------------------
#     # 5) CTC decode for text comparison
#     # ---------------------------------------

#     decoded_ids = ctc_decode(frame_preds, blank_id)

#     decoded_phonemes = [IDX2PHONEME[i] for i in decoded_ids]

#     predicted_text = phonemes_to_text(decoded_phonemes)

#     # ---------------------------------------
#     # 6) Extract Tajweed rules from ayah text
#     # ---------------------------------------

#     phoneme_seq, phoneme_meta, madd_rules = extract_madd_rules(ayah_text)

#     # phoneme_seq = expected phoneme sequence

#     # ---------------------------------------
#     # 7) Align expected vs predicted phonemes
#     # ---------------------------------------

#     alignment = align_phonemes(phoneme_seq, predicted_phonemes)

#     # alignment example:
#     # {0:0, 1:None, 2:1, 3:2}

#     # ---------------------------------------
#     # 8) Evaluate Madd rules
#     # ---------------------------------------

#     madd_results = []
#     madd_errors = []

#     avg_short = compute_avg_short_duration(segments)

#     if avg_short is not None:

#         for rule in madd_rules:

#             expected_idx = rule["phoneme_index"]
#             expected_harakat = rule["expected_harakat"]

#             pred_idx = alignment.get(expected_idx)

#             if pred_idx is None:
#                 continue

#             if pred_idx >= len(segments):
#                 continue

#             pid, frames = segments[pred_idx]

#             observed = frames / avg_short

#             if abs(observed - expected_harakat) <= 0.8:
#                 status = "Correct"
#             elif observed < expected_harakat:
#                 status = "Too Short"
#             else:
#                 status = "Too Long"

#             result = {
#                 "phoneme_index": expected_idx,
#                 "type": rule["type"],
#                 "expected": expected_harakat,
#                 "observed": round(observed, 2),
#                 "status": status,
#             }

#             madd_results.append(result)

#             if status != "Correct":
#                 madd_errors.append(result)

#     # ---------------------------------------
#     # 9) Word error analysis
#     # ---------------------------------------

#     word_error_list = word_errors(ayah_text, predicted_text)

#     # ---------------------------------------
#     # 10) Debug info (very useful)
#     # ---------------------------------------

#     debug = {
#         "expected_phonemes": phoneme_seq,
#         "predicted_phonemes": predicted_phonemes,
#         "segments": segments,
#         "alignment": alignment,
#     }

#     # ---------------------------------------
#     # 11) Final evaluation result
#     # ---------------------------------------

#     result = {
#         "expected_text": ayah_text,
#         "predicted_text": predicted_text,
#         "word_errors": word_error_list,
#         "madd_results": madd_results,
#         "madd_errors": madd_errors,
#         "debug": debug,
#     }

#     return result

In [ ]:
# print(df.iloc[2]["ayah_text"])

بِسْمِ اللَّهِ الرَّحْمَـٰنِ الرَّحِيمِ الْحَمْدُ لِلَّهِ الَّذِي أَنزَلَ عَلَىٰ عَبْدِهِ الْكِتَابَ وَلَمْ يَجْعَل لَّهُ عِوَجًا ۜ


In [ ]:
# evaluate_audio(
#     model=model,
#     audio_path=df.iloc[1]["path_of_audio"],
#     ayah_text=df.iloc[1]["ayah_text"],
# )

{'expected_text': 'لَا يُكَلِّفُ اللَّهُ نَفْسًا إِلَّا وُسْعَهَا ۚ لَهَا مَا كَسَبَتْ وَعَلَيْهَا مَا اكْتَسَبَتْ ۗ رَبَّنَا لَا تُؤَاخِذْنَا إِن نَّسِينَا أَوْ أَخْطَأْنَا ۚ رَبَّنَا وَلَا تَحْمِلْ عَلَيْنَا إِصْرًا كَمَا حَمَلْتَهُ عَلَى الَّذِينَ مِن قَبْلِنَا ۚ رَبَّنَا وَلَا تُحَمِّلْنَا مَا لَا طَاقَةَ لَنَا بِهِ ۖ وَاعْفُ عَنَّا وَاغْفِرْ لَنَا وَارْحَمْنَا ۚ أَنتَ مَوْلَانَا فَانصُرْنَا عَلَى الْقَوْمِ الْكَافِرِينَ',
 'predicted_text': 'ءِفِسُفِلُءِدَفٌلُءِلُءِفٌءِلُدَفٌفِدَءِلُءِدَلُءِفٌءِلُءِلُءِلُفِفٌءِفِءِدَءِفِءِفِءِفِلُفِءِفِءِفِءِفِءِفِءِفِلُفِءِفِءِفِءِلُفِءِفِلُءِدَءِلُءِلُءِخًءِبُفِءِفِلُءِفِلُفِءِدَفِءِلُفِلُفِدَفِلُءِتَفٌءِلُءِلُفٌءِلُءِدَلُءِلُءِلُءِلُءِلُءِلُءِفٌفِفٌءِدَءِفٌءِفٌفِلُءِلُءِلُءِفٌءِلُءِلُءِلُفِءِلُءِلُءِلُءِلُءِلُءِفِءِلُءِلُءِلُءِلُءِلُءِفِءِفِءِلُءِدَءِفِفٌءِدَءِدَفِءِفٌفِءِفِلُءِفِلُفِءِلُءِلُءِفِلُءِلُءِفِءِلُءِضِيلُسُهُوءِفِلُءِضِيءِلُءِلُدَفٌءِلُءِلُبُسُفٌفِلُءِدَءِلُبُخُوفِضِيلُءِدَلُءِفِءِفِءِلُدَءِلُفِءِلُفِءِفِءِبُفٌفِلُفِءِدَتَءِفِلُفِءِ